# Hospital Readmission Analysis - Data Preprocessing

This notebook prepares the raw diabetes hospital readmission dataset for modeling. Based on the data understanding step, we will handle inconsistent missing values, remove columns with very high missingness, and begin transforming categorical features into a cleaner format.

In [1]:
import pandas as pd
import numpy as np

## Loading the Raw Dataset

We start from the original raw dataset to make sure all preprocessing steps are reproducible and not dependent on changes made in the previous notebook.

In [2]:
df = pd.read_csv("../data/raw/diabetic_data.csv")
df.head(3)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## Initial Dataset Shape

Before making changes, we check the size of the dataset. This helps confirm that we are starting from the original raw file.

In [3]:
df.shape

(101766, 50)

## Converting Inconsistent Missing Values

During data understanding, we found that some missing values are stored as the string `"?"` instead of standard missing values. These need to be converted to `NaN` so that pandas can correctly recognize and handle them.

In [4]:
df = df.replace("?", pd.NA)

## Verifying Missing Value Conversion

After replacing `"?"`, we check whether any question-mark placeholders remain in the dataset.

In [5]:
(df == "?").sum().sum()

np.int64(0)

## Dropping High-Missing Columns

From the data understanding notebook, the following columns were identified as having very high missingness or limited usefulness for this prediction task:

- `weight`
- `payer_code`
- `max_glu_serum`
- `A1Cresult`

These columns are removed here as part of preprocessing.

In [6]:
cols_to_drop = ["weight", "payer_code", "max_glu_serum", "A1Cresult"]

df = df.drop(columns=cols_to_drop)

df.shape

(101766, 46)

## Handling `medical_specialty`

The `medical_specialty` column has many categories and about half of its values are missing. Keeping every specialty as a separate category would create a sparse feature. To keep the feature useful and simple, we retain the most common specialties, group rare specialties as `Other`, and label missing values as `Unknown`.

In [7]:
top_5_specialties = df["medical_specialty"].value_counts().head(5).index

top_5_specialties

Index(['InternalMedicine', 'Emergency/Trauma', 'Family/GeneralPractice',
       'Cardiology', 'Surgery-General'],
      dtype='str', name='medical_specialty')

## Creating a Grouped Specialty Feature

We create a new column called `medical_specialty_grouped` instead of overwriting the original column. This keeps the original feature available for reference while giving us a cleaner version for modeling.

In [8]:
def group_medical_specialty(value):
    if pd.isna(value):
        return "Unknown"
    elif value in top_5_specialties:
        return value
    else:
        return "Other"

df["medical_specialty_grouped"] = df["medical_specialty"].apply(group_medical_specialty)

df["medical_specialty_grouped"].value_counts()

medical_specialty_grouped
Unknown                   49949
InternalMedicine          14635
Other                     13726
Emergency/Trauma           7565
Family/GeneralPractice     7440
Cardiology                 5352
Surgery-General            3099
Name: count, dtype: int64

## Specialty Grouping Check

The grouped column now reduces many low-frequency specialties into a single `Other` category while preserving the most common specialties. Missing values are represented as `Unknown`, so the model can still use missingness as possible information instead of losing rows.

## Creating the Binary Target Variable

The original `readmitted` column has three values: `NO`, `>30`, and `<30`. Since this project focuses on predicting 30-day readmission, we convert it into a binary target where `<30` is treated as readmitted within 30 days and all other cases are treated as not readmitted within 30 days.

In [9]:
df["readmitted_30_days"] = df["readmitted"].apply(lambda x: 1 if x == "<30" else 0)

df["readmitted_30_days"].value_counts(normalize=True) * 100

readmitted_30_days
0    88.840084
1    11.159916
Name: proportion, dtype: float64

## Removing Columns Not Needed for Modeling

Some columns are identifiers or original versions of features that are no longer needed for modeling. We remove these columns to keep the modeling dataset cleaner.

In [10]:
cols_to_remove = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "medical_specialty"
]

df = df.drop(columns=cols_to_remove)

df.shape

(101766, 44)

## Checking Remaining Missing Values

After the major preprocessing steps, we check the remaining missing values to decide what needs to be handled before encoding and modeling.

In [11]:
missing_remaining = df.isna().sum()
missing_remaining[missing_remaining > 0].sort_values(ascending=False)

race      2273
diag_3    1423
diag_2     358
diag_1      21
dtype: int64

## Handling Remaining Missing Values

After preprocessing, a few columns still contain missing values. The remaining missing values are in categorical fields such as race and diagnosis codes. Since the missing percentages are small, these values are labeled as `Unknown` instead of dropping rows.

In [14]:
cols_fill_unknown = ["race", "diag_1", "diag_2", "diag_3"]

df[cols_fill_unknown] = df[cols_fill_unknown].fillna("Unknown")

df.isna().sum().sum()

np.int64(0)

## Saving the Preprocessed Dataset

The cleaned dataset is saved to the processed data folder so that later notebooks can start from this prepared version instead of repeating the same preprocessing steps.

In [15]:
df.to_csv("../data/processed/diabetic_data_preprocessed.csv", index=False)

## Preprocessing Summary

In this notebook, inconsistent missing values were converted to proper missing values, high-missing columns were removed, `medical_specialty` was grouped into broader categories, and the target variable was converted into a binary outcome for 30-day readmission prediction. The dataset is now ready for categorical encoding and model preparation.